# **Prediction Random Forest**

In [ ]:
LOCAL = False
USER = "default"

In [ ]:
scenarios = {
    # Baseline
    "baseline": {
        "executor_instances": 2,
        "executor_cores": 1,
        "executor_memory": "3g",
        "shuffle_partitions": 50,
        "adaptive": False
    },

    # Balanced
    "balanced": {
        "executor_instances": 2,
        "executor_cores": 2,
        "executor_memory": "4g",
        "shuffle_partitions": 100,
        "adaptive": True
    },

    # High parallelism
    "high_parallelism": {
        "executor_instances": 3,
        "executor_cores": 2,
        "executor_memory": "4g",
        "shuffle_partitions": 150,
        "adaptive": True
    },

    # Memory optimized
    "memory_optimized": {
        "executor_instances": 2,
        "executor_cores": 1,
        "executor_memory": "6g",
        "shuffle_partitions": 100,
        "adaptive": True
    },

    # Stress test
    "stress_test": {
        "executor_instances": 4,
        "executor_cores": 2,
        "executor_memory": "3g",
        "shuffle_partitions": 200,
        "adaptive": True
    },

    # Low overhead
    "low_overhead": {
        "executor_instances": 2,
        "executor_cores": 2,
        "executor_memory": "4g",
        "shuffle_partitions": 25,
        "adaptive": False
    },

    # AQE isolation test
    "aqe_only": {
        "executor_instances": 2,
        "executor_cores": 1,
        "executor_memory": "3g",
        "shuffle_partitions": 50,
        "adaptive": True
    },

    # CPU vs memory balanced stress
    "cpu_heavy": {
        "executor_instances": 3,
        "executor_cores": 3,
        "executor_memory": "3g",
        "shuffle_partitions": 150,
        "adaptive": True
    }
}

MODE = "cpu_heavy"

**Libraries**

In [4]:
import glob
import os
import pyarrow.parquet as pq
from pyspark.sql.functions import broadcast
import pandas as pd
import numpy as np
import seaborn as sns
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, count, when, isnan,
    countDistinct,
    sum as _sum,
    unix_timestamp,
    count as spark_count,
    abs as spark_abs,
    hour, dayofweek, month, dayofmonth, weekofyear, to_date, date_trunc,
    avg, lower, desc, rank, trim, isnull, percentile_approx,
    concat, lit,
    max as spark_max,
    year
)
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, TimestampType
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.patches import FancyArrowPatch
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec

import math
import geopandas as gpd
import time

/opt/conda/miniconda3/lib/python3.10/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.26.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


**Spark Session**

In [5]:
from pyspark.sql import SparkSession

if LOCAL:
  spark = SparkSession.builder.master("local[*]").appName("NYC_Taxi_Analysis").config("spark.driver.memory", "8g").getOrCreate()

  print("Local")
else:
  conf = scenarios[MODE]

  spark = (
      SparkSession.builder
      .appName("NYC_Taxi_Analysis")

      # Spark tuning
      .config("spark.sql.shuffle.partitions", str(conf["shuffle_partitions"]))
      .config("spark.sql.adaptive.enabled", str(conf["adaptive"]).lower())

      # Executor-level tuning (Dataproc/YARN)
      .config("spark.executor.instances", str(conf["executor_instances"]))
      .config("spark.executor.cores", str(conf["executor_cores"]))
      .config("spark.executor.memory", conf["executor_memory"])

      .getOrCreate()
  )

  print(f"Mode: {MODE}")
  print(f"Executor instances: {conf['executor_instances']}")
  print(f"Executor cores: {conf['executor_cores']}")
  print(f"Executor memory: {conf['executor_memory']}")
  print(f"Shuffle partitions: {conf['shuffle_partitions']}")
  print(f"Adaptive execution: {conf['adaptive']}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/19 19:19:16 INFO SparkEnv: Registering MapOutputTracker
26/05/19 19:19:16 INFO SparkEnv: Registering BlockManagerMaster
26/05/19 19:19:16 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/05/19 19:19:16 INFO SparkEnv: Registering OutputCommitCoordinator


Mode: cpu_heavy
Executor instances: 2
Executor cores: 2
Executor memory: 3g
Shuffle partitions: 150
Adaptive execution: True


**Create taxi_df**

In [6]:
# Data path
if LOCAL:
  DATA_PATH = "./data/2020"
else:
  if USER == "mariana":
    DATA_PATH = "gs://nyc-taxi-data-27/"
  elif USER == "patricia":
    DATA_PATH = "gs://taxi-data-2020-patricia/"
  elif USER == "leonor":
    DATA_PATH = "gs://taxi-data-2020-egd/"
  elif USER == "luana":
    DATA_PATH = "gs://egd_feup/"
  elif USER == "iara":
    DATA_PATH = "gs://taxi-data-2020i/"
  else:
    raise ValueError("Unknown user. Please set the DATA_PATH variable accordingly.")

# Parquet files logic
if LOCAL:
  parquet_files = glob.glob(
    os.path.join(DATA_PATH, "*.parquet")
  )
  print(f"\nFound {len(parquet_files)} parquet files.")

  # Count total rows
  total_rows = 0
  for file in parquet_files:
      parquet_file = pq.ParquetFile(file)
      rows = parquet_file.metadata.num_rows
      print(f"{os.path.basename(file)} -> {rows:,} rows")
      total_rows += rows

  print("\n=========================")
  print(f"TOTAL ROWS: {total_rows:,}")
  print("=========================")
else:
  parquet_path = f"{DATA_PATH}/*.parquet"

# 1. Define explicitly the type of each column (Fixed: Using TimestampType instead of TimestampNTZType)
taxi_schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", DoubleType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True)
])

spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")

# 2. Read files
if LOCAL:
  taxi_df = spark.read.schema(taxi_schema).parquet(*parquet_files)
else:
  taxi_df = spark.read.schema(taxi_schema).parquet(parquet_path)

print("\nDataset loaded successfully with explicit schema!")

# Dataset info
print("\n=========================")
print("TOTAL ROWS:", taxi_df.count())
print("TOTAL COLUMNS:", len(taxi_df.columns))
print("=========================")

# Load taxi_zone_lookup.csv
zone_df = spark.read.csv(
    f"{DATA_PATH}/taxi_zone_lookup.csv",
    header=True,
    inferSchema=True
)

print("\nTaxi Zone Lookup Preview:")
zone_df.show(5)

# Pickup join
pickup_lookup = (
    zone_df
    .withColumnRenamed("LocationID", "PULocationID")
    .withColumnRenamed("Borough", "PU_Borough")
    .withColumnRenamed("Zone", "PU_Zone")
    .withColumnRenamed("service_zone", "PU_service_zone")
)

taxi_df = taxi_df.join(
    broadcast(pickup_lookup),
    on="PULocationID",
    how="left"
)

print("\nPickup join completed!")

# Dropoff join
dropoff_lookup = (
    zone_df
    .withColumnRenamed("LocationID", "DOLocationID")
    .withColumnRenamed("Borough", "DO_Borough")
    .withColumnRenamed("Zone", "DO_Zone")
    .withColumnRenamed("service_zone", "DO_service_zone")
)

taxi_df = taxi_df.join(
    broadcast(dropoff_lookup),
    on="DOLocationID",
    how="left"
)

print("\nDropoff join completed!")

# Final dataset info
print("\nFinal Schema:")
taxi_df.printSchema()

print("\nFinal Dataset Preview:")
taxi_df.show(5)


Dataset loaded successfully with explicit schema!



TOTAL ROWS: 24649092
TOTAL COLUMNS: 19



Taxi Zone Lookup Preview:
+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


Pickup join completed!

Dropoff join completed!

Final Schema:
root
 |-- DOLocationID: long (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- 

+------------+------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------+--------------------+---------------+----------+--------------------+---------------+
|DOLocationID|PULocationID|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|PU_Borough|             PU_Zone|PU_service_zone|DO_Borough|             DO_Zone|DO_service_zone|
+------------+------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+--------

In [7]:
n_rows = taxi_df.count()
n_cols = len(taxi_df.columns)

print("=" * 50)
print("DATASET STRUCTURE")
print("=" * 50)
print(f"Number of rows    : {n_rows:,}")
print(f"Number of columns : {n_cols}")
print(f"Granularity       : 1 row = 1 completed taxi trip")
print("=" * 50)

# Files & disk size
if LOCAL:
    total_size_bytes = sum(os.path.getsize(f) for f in parquet_files)
    print(f"Number of files   : {len(parquet_files)}")
    print(f"Total size on disk: {total_size_bytes / (1024**2):.2f} MB "
          f"({total_size_bytes / (1024**3):.2f} GB)")

DATASET STRUCTURE
Number of rows    : 24,649,092
Number of columns : 25
Granularity       : 1 row = 1 completed taxi trip


### **3.3. Verify Data Quality**

#### **3.3.1. Time Inconsistencies**

In [8]:
taxi_df = taxi_df.filter(
    (F.col("tpep_pickup_datetime") >= "2020-01-01") &
    (F.col("tpep_pickup_datetime") <= "2020-12-31")
)

taxi_df.select(
    F.min("tpep_pickup_datetime").alias("start_date"),
    F.max("tpep_pickup_datetime").alias("end_date")
).show()

+-------------------+-------------------+
|         start_date|           end_date|
+-------------------+-------------------+
|2020-01-01 00:00:00|2020-12-31 00:00:00|
+-------------------+-------------------+



The dataset was filtered to retain only records within the year 2020. A total of **45,072 records** (0.18% of the dataset) presented `tpep_pickup_datetime` values outside this range — either predating 2020 or extending beyond December 31, 2020 — and were removed as 
invalid entries.

#### **3.3.4. Missing values**

In [9]:
bad_strings = ['nan', 'null', 'n/a', 'na', '', 'unknown']

string_cols = [c for c, t in taxi_df.dtypes if t == 'string']

for c in string_cols:
    taxi_df = taxi_df.withColumn(
        c,
        when(lower(trim(col(c))).isin(bad_strings), lit(None).cast("string"))
        .otherwise(col(c))
    )

print("Conversion completed! Verifying results...")


print("\nNulls after cleaning text columns:")
taxi_df.select([
    _sum(when(isnull(col(c)), 1).otherwise(0)).alias(c)
    for c in string_cols
]).show(vertical=True)

Conversion completed! Verifying results...

Nulls after cleaning text columns:


-RECORD 0--------------------
 store_and_fwd_flag | 806563 
 PU_Borough         | 223831 
 PU_Zone            | 57481  
 PU_service_zone    | 223831 
 DO_Borough         | 208937 
 DO_Zone            | 56807  
 DO_service_zone    | 208937 



In [10]:
print("Starting Data Cleaning Pipeline...")
total_rows = taxi_df.count()

# 100% Missing
taxi_df = taxi_df.drop("airport_fee")

# ~3.29% Missing
taxi_df = taxi_df \
    .withColumn(
        "RatecodeID",
        when(col("RatecodeID").isNull(),
             # If origin or destination is JFK (Zone ID 132 is JFK), apply RateCode 2
             when((col("PULocationID") == 132) | (col("DOLocationID") == 132), 2.0)
             # Otherwise, apply the standard fare (1)
             .otherwise(1.0)
        ).otherwise(col("RatecodeID"))
    ) \
    .withColumn(
        "congestion_surcharge",
        when(col("congestion_surcharge").isNull(),
             # If started or ended in Manhattan, apply the $2.50 fee
             when((col("PU_Borough") == 'Manhattan') | (col("DO_Borough") == 'Manhattan'), 2.5)
             .otherwise(0.0)
        ).otherwise(col("congestion_surcharge"))
    ) \
    .fillna({
        "passenger_count": 1.0,
        "store_and_fwd_flag": "N"
    })

# < 1% Missing
# Drop rows where critical geographic or categorical data is still missing
geo_cols_to_check = [
    "PU_Borough", "PU_service_zone", "PU_Zone",
    "DO_Borough", "DO_service_zone", "DO_Zone"
]

taxi_df = taxi_df.dropna(subset=geo_cols_to_check)

print("Data Cleaning and Imputation completed!")

# Validation
check_cols = [
    "passenger_count", "RatecodeID", "store_and_fwd_flag",
    "congestion_surcharge", "PU_Borough", "DO_Borough"
]

print("\nFinal Null Count Check (Should be all 0):")
taxi_df.select([
    _sum(col(c).isNull().cast("int")).alias(c)
    for c in check_cols
]).show(vertical=True)

Starting Data Cleaning Pipeline...


Data Cleaning and Imputation completed!

Final Null Count Check (Should be all 0):


-RECORD 0-------------------
 passenger_count      | 0   
 RatecodeID           | 0   
 store_and_fwd_flag   | 0   
 congestion_surcharge | 0   
 PU_Borough           | 0   
 DO_Borough           | 0   



#### **3.3.5. Global Inconsistencies**

In [11]:
raw_row_count = taxi_df.count()
print(f"Initial row count: {raw_row_count:,}\n")

# -- Helper: trip duration in minutes (temporary, dropped at the end) --
taxi_df = taxi_df.withColumn(
    "duration_min",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0
)

# ------------------------------------------------------------
# STEP 1 — Detect inconsistencies (report counts before fixing)
# ------------------------------------------------------------
checks = {
    # Duplicates
    "Exact duplicate rows":
        raw_row_count - taxi_df.distinct().count(),

    # Temporal anomalies
    "Dropoff before pickup":
        taxi_df.filter(F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime")).count(),
    "Duration < 1 minute":
        taxi_df.filter(F.col("duration_min") < 1).count(),
    "Duration > 6 hours":
        taxi_df.filter(F.col("duration_min") > 360).count(),
    "Pickup outside 2020":
        taxi_df.filter(F.year("tpep_pickup_datetime") != 2020).count(),

    # Distance anomalies
    "trip_distance <= 0":
        taxi_df.filter(F.col("trip_distance") <= 0).count(),
    "trip_distance > 100 miles":
        taxi_df.filter(F.col("trip_distance") > 100).count(),

    # Speed anomalies (derived)
    "Avg speed > 100 mph":
        taxi_df.filter(
            (F.col("duration_min") > 1) &
            ((F.col("trip_distance") / (F.col("duration_min") / 60)) > 100)
        ).count(),

    # Financial anomalies
    "fare_amount < 2.5 (below NYC initial charge)":
        taxi_df.filter(F.col("fare_amount") < 2.5).count(),
    "fare_amount > 500":
        taxi_df.filter(F.col("fare_amount") > 500).count(),
    "total_amount <= 0":
        taxi_df.filter(F.col("total_amount") <= 0).count(),
    "total_amount > 600":
        taxi_df.filter(F.col("total_amount") > 600).count(),
    "tip_amount < 0":
        taxi_df.filter(F.col("tip_amount") < 0).count(),
    "total_amount != sum of components (tol. 0.1)":
        taxi_df.filter(
            F.abs(
                F.col("total_amount") - (
                    F.col("fare_amount") + F.col("extra") + F.col("mta_tax") +
                    F.col("tip_amount") + F.col("tolls_amount") +
                    F.col("improvement_surcharge") + F.coalesce(F.col("congestion_surcharge"), F.lit(0))
                )
            ) > 0.1
        ).count(),

    # Passenger count
    "passenger_count = 0":
        taxi_df.filter(F.col("passenger_count") == 0).count(),
    "passenger_count > 6":
        taxi_df.filter(F.col("passenger_count") > 6).count(),

    # Invalid categorical codes
    "Invalid RatecodeID (not in 1..6)":
        taxi_df.filter(~F.col("RatecodeID").isin([1, 2, 3, 4, 5, 6])).count(),
    "Invalid PULocationID (not in 1..263)":
        taxi_df.filter(~F.col("PULocationID").between(1, 263)).count(),
    "Invalid DOLocationID (not in 1..263)":
        taxi_df.filter(~F.col("DOLocationID").between(1, 263)).count(),
}

# Build report
import pandas as pd
quality_df = pd.DataFrame(list(checks.items()), columns=["Inconsistency", "Count"])
quality_df["% of total"] = (100 * quality_df["Count"] / raw_row_count).round(3)
quality_df = quality_df.sort_values("Count", ascending=False).reset_index(drop=True)

print("=" * 75)
print("DATA INCONSISTENCIES DETECTED")
print("=" * 75)
print(quality_df.to_string(index=False))

# ------------------------------------------------------------
# STEP 2 — Apply cleaning rules
# ------------------------------------------------------------
taxi_clean = (
    taxi_df
    # 2.1 — Deduplicate
    .dropDuplicates()

    # 2.2 — Temporal validity
    .filter(F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
    .filter((F.col("duration_min") >= 1) & (F.col("duration_min") <= 360))
    .filter(F.year("tpep_pickup_datetime") == 2020)

    # 2.3 — Distance validity
    .filter((F.col("trip_distance") > 0) & (F.col("trip_distance") <= 100))

    # 2.4 — Speed sanity check
    .filter((F.col("trip_distance") / (F.col("duration_min") / 60)) < 100)

    # 2.5 — Financial validity
    .filter((F.col("fare_amount") >= 2.5) & (F.col("fare_amount") <= 500))
    .filter((F.col("total_amount") > 0) & (F.col("total_amount") <= 600))
    .filter(F.col("tip_amount") >= 0)

    # 2.6 — Passenger count
    .filter((F.col("passenger_count") >= 1) & (F.col("passenger_count") <= 6))

    # 2.7 — Valid categorical codes
    .filter(F.col("RatecodeID").isin([1, 2, 3, 4, 5, 6]))
    .filter(F.col("PULocationID").between(1, 263))
    .filter(F.col("DOLocationID").between(1, 263))

    # 2.8 — Drop helper column
    .drop("duration_min")
)

# Cache + register view for downstream queries
taxi_clean.cache()
taxi_clean.createOrReplaceTempView("taxi_clean")

clean_count = taxi_clean.count()
removed = raw_row_count - clean_count
retention = 100 * clean_count / raw_row_count

taxi_df = taxi_clean

print("\n" + "=" * 75)
print("CLEANING RESULTS")
print("=" * 75)
print(f"Rows before cleaning : {raw_row_count:,}")
print(f"Rows after cleaning  : {clean_count:,}")
print(f"Rows removed         : {removed:,} ({100 - retention:.2f}%)")
print(f"Retention rate       : {retention:.2f}%")
print(f"Final column count   : {len(taxi_clean.columns)}")
print("=" * 75)

Initial row count: 24,285,406



26/05/19 19:20:41 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DATA INCONSISTENCIES DETECTED
                               Inconsistency   Count  % of total
total_amount != sum of components (tol. 0.1) 7532187      31.015
                         passenger_count = 0  482390       1.986
                          trip_distance <= 0  282114       1.162
                         Duration < 1 minute  207062       0.853
fare_amount < 2.5 (below NYC initial charge)  100846       0.415
                           total_amount <= 0   95288       0.392
                          Duration > 6 hours   47189       0.194
                        Exact duplicate rows   12841       0.053
                         Avg speed > 100 mph    1176       0.005
                              tip_amount < 0     863       0.004
            Invalid RatecodeID (not in 1..6)     791       0.003
                   trip_distance > 100 miles     246       0.001
                       Dropoff before pickup     194       0.001
                         passenger_count > 6     110       0


CLEANING RESULTS
Rows before cleaning : 24,285,406
Rows after cleaning  : 23,304,000
Rows removed         : 981,406 (4.04%)
Retention rate       : 95.96%
Final column count   : 24


### **3.4. Feature Engineering**

In [12]:
print("Starting feature engineering...")

taxi_features = (
    taxi_df

    # --- Temporal features (derived from pickup timestamp) ---
    .withColumn("pickup_hour",        F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_day_of_week", F.dayofweek("tpep_pickup_datetime"))  # 1=Sun, 7=Sat
    .withColumn("pickup_month",       F.month("tpep_pickup_datetime"))
    .withColumn("pickup_date",        F.to_date("tpep_pickup_datetime"))

    # --- Operational metrics ---
    .withColumn(
        "trip_duration_min",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0
    )
    .withColumn(
        "speed_mph",
        F.when(
            F.col("trip_duration_min") > 0,
            F.col("trip_distance") / (F.col("trip_duration_min") / 60.0)
        )
    )
    .withColumn(
        "fare_per_mile",
        F.when(F.col("trip_distance") > 0, F.col("fare_amount") / F.col("trip_distance"))
    )

    # --- Behavioural feature: tip percentage (credit-card only) ---
    # Cash tips are not recorded by the meter, so we restrict to payment_type = 1
    .withColumn(
        "tip_pct",
        F.when(
            (F.col("payment_type") == 1) & (F.col("fare_amount") > 0),
            (F.col("tip_amount") / F.col("fare_amount")) * 100
        ).otherwise(0.0)
    )

    # --- Temporal flags ---
    # Rush hour: weekday mornings (7–9) and evenings (16–19)
    .withColumn(
        "is_rush_hour",
        (
            (F.col("pickup_day_of_week").between(2, 6)) &  # Mon–Fri
            (
                F.col("pickup_hour").between(7, 9) |
                F.col("pickup_hour").between(16, 19)
            )
        )
    )
    .withColumn(
        "is_weekend",
        F.col("pickup_day_of_week").isin([1, 7])  # Sun or Sat
    )

    # --- Time-of-day bucket (for visual aggregations) ---
    .withColumn(
        "time_of_day",
        F.when(F.col("pickup_hour").between(0, 5), "Night")
         .when(F.col("pickup_hour").between(6, 11), "Morning")
         .when(F.col("pickup_hour").between(12, 17), "Afternoon")
         .otherwise("Evening")
    )

    # --- Human-readable categorical labels ---
    .withColumn(
        "payment_name",
        F.when(F.col("payment_type") == 1, "Credit Card")
         .when(F.col("payment_type") == 2, "Cash")
         .when(F.col("payment_type") == 3, "No Charge")
         .when(F.col("payment_type") == 4, "Dispute")
         .when(F.col("payment_type") == 5, "Unknown")
         .when(F.col("payment_type") == 6, "Voided")
         .otherwise("Other")
    )
    .withColumn(
        "vendor_name",
        F.when(F.col("VendorID") == 1, "Creative Mobile")
         .when(F.col("VendorID") == 2, "VeriFone")
         .otherwise("Other")
    )
    .withColumn(
        "rate_name",
        F.when(F.col("RatecodeID") == 1, "Standard")
         .when(F.col("RatecodeID") == 2, "JFK")
         .when(F.col("RatecodeID") == 3, "Newark")
         .when(F.col("RatecodeID") == 4, "Nassau/Westchester")
         .when(F.col("RatecodeID") == 5, "Negotiated")
         .when(F.col("RatecodeID") == 6, "Group Ride")
         .otherwise("Other")
    )
)

# Cache the enriched DataFrame — every downstream query reuses these features
taxi_features.cache()
taxi_features.createOrReplaceTempView("taxi_features")
taxi_df = taxi_features

print(f"Feature engineering complete.")
print(f"Total columns after feature engineering: {len(taxi_features.columns)}")
print(f"\nNew features added:")
print("  Temporal     : pickup_hour, pickup_day_of_week, pickup_month, pickup_date")
print("  Operational  : trip_duration_min, speed_mph, fare_per_mile")
print("  Behavioural  : tip_pct")
print("  Flags        : is_rush_hour, is_weekend, time_of_day")
print("  Labels       : payment_name, vendor_name, rate_name")

Starting feature engineering...
Feature engineering complete.
Total columns after feature engineering: 38

New features added:
  Temporal     : pickup_hour, pickup_day_of_week, pickup_month, pickup_date
  Operational  : trip_duration_min, speed_mph, fare_per_mile
  Behavioural  : tip_pct
  Flags        : is_rush_hour, is_weekend, time_of_day
  Labels       : payment_name, vendor_name, rate_name


---

# **5. ML Prediction**

In [13]:
import time
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)
from pyspark.sql.functions import col
from pyspark.ml.feature import StandardScaler

from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [14]:
model_results = []

### **5.1. Total Amount**

In [15]:
start_time = time.time()

# DATA PREPARATION

amount_df = taxi_df.filter(
    col("total_amount").isNotNull()
)

amount_target = "total_amount"

# FEATURES

numeric_features = [
    "trip_distance",
    "passenger_count",
    "pickup_hour",
    "speed_mph",
    "fare_amount",
    "tip_amount"
]

categorical_features = [
    "PU_Borough",
    "DO_Borough",
    "time_of_day",
]

# STRING INDEXERS

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep"
    )
    for c in categorical_features
]

# ONE HOT ENCODING

encoders = [
    OneHotEncoder(
        inputCol=f"{c}_idx",
        outputCol=f"{c}_ohe"
    )
    for c in categorical_features
]

# FEATURE VECTOR

assembler_inputs = numeric_features + [
    f"{c}_ohe" for c in categorical_features
]

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features"
)

# MODEL

amount_model = RandomForestRegressor(
    featuresCol="features",
    labelCol=amount_target,
    predictionCol="prediction",
    numTrees=25,
    maxDepth=8,
    minInstancesPerNode=5000,
    minInfoGain=0.05,
    maxBins=128,
    subsamplingRate=0.8,
    featureSubsetStrategy="sqrt",
    seed=42
)

# PIPELINE

amount_pipeline = Pipeline(stages=
    indexers +
    encoders +
    [assembler, amount_model]
)

# TRAIN / TEST SPLIT

amount_train, amount_test = amount_df.randomSplit([0.8, 0.2], seed=42)

amount_train.cache()

# TRAINING

amount_fitted_model = amount_pipeline.fit(amount_train)

# PREDICTIONS

amount_predictions = amount_fitted_model.transform(amount_test)

# METRICS

amount_rmse = RegressionEvaluator(
    labelCol=amount_target,
    predictionCol="prediction",
    metricName="rmse"
).evaluate(amount_predictions)

amount_r2 = RegressionEvaluator(
    labelCol=amount_target,
    predictionCol="prediction",
    metricName="r2"
).evaluate(amount_predictions)

amount_mae = RegressionEvaluator(
    labelCol=amount_target,
    predictionCol="prediction",
    metricName="mae"
).evaluate(amount_predictions)

amount_mse = RegressionEvaluator(
    labelCol=amount_target,
    predictionCol="prediction",
    metricName="mse"
).evaluate(amount_predictions)

amount_execution_time = time.time() - start_time


Save to the model_results list:

In [16]:
model_results.append({
    "model_name": "RandomForestRegressor",
    "target": "total_amount",
    "task_type": "regression",
    "metrics": {
        "rmse": round(amount_rmse, 4),
        "r2": round(amount_r2, 4),
        "mae": round(amount_mae, 4),
        "mse": round(amount_mse, 4)
    },
    "execution_time_seconds": round(amount_execution_time, 2)
})

print("\nTotal amount prediction completed!")



Total amount prediction completed!


### **5.2. Trip Duration**

In [17]:
start_time = time.time()

# DATA PREPARATION
duration_df = taxi_df.filter(
    col("trip_duration_min").isNotNull()
)

duration_target = "trip_duration_min"

# FEATURES

numeric_features = [
    "trip_distance",
    "passenger_count",
    "pickup_hour",
    "pickup_day_of_week",
    "fare_amount",
]

categorical_features = [
    "PU_Borough",
    "DO_Borough",
    "time_of_day",
]

# STRING INDEXERS

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep"
    )
    for c in categorical_features
]

# ONE HOT ENCODING

encoders = [
    OneHotEncoder(
        inputCol=f"{c}_idx",
        outputCol=f"{c}_ohe"
    )
    for c in categorical_features
]

# FEATURE VECTOR

assembler_inputs = numeric_features + [
    f"{c}_ohe" for c in categorical_features
]

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features"
)

# MODEL

duration_model = RandomForestRegressor(
    featuresCol="features",
    labelCol=duration_target,
    predictionCol="prediction",
    numTrees=25,
    maxDepth=8,
    minInstancesPerNode=5000,
    minInfoGain=0.05,
    maxBins=128,
    subsamplingRate=0.8,
    featureSubsetStrategy="sqrt",
    seed=42
)

# PIPELINE

duration_pipeline = Pipeline(stages=
    indexers +
    encoders +
    [assembler, duration_model]
)

# TRAIN / TEST SPLIT

duration_train, duration_test = duration_df.randomSplit([0.8, 0.2], seed=42)

duration_train.cache()

# TRAINING

duration_model_fitted = duration_pipeline.fit(duration_train)

# PREDICTIONS

duration_predictions = duration_model_fitted.transform(duration_test)

duration_rmse = RegressionEvaluator(
    labelCol=duration_target,
    predictionCol="prediction",
    metricName="rmse"
).evaluate(duration_predictions)

duration_r2 = RegressionEvaluator(
    labelCol=duration_target,
    predictionCol="prediction",
    metricName="r2"
).evaluate(duration_predictions)

duration_mae = RegressionEvaluator(
    labelCol=duration_target,
    predictionCol="prediction",
    metricName="mae"
).evaluate(duration_predictions)

duration_mse = RegressionEvaluator(
    labelCol=duration_target,
    predictionCol="prediction",
    metricName="mse"
).evaluate(duration_predictions)

duration_execution_time = time.time() - start_time


Save to the model_results list:

In [18]:
model_results.append({
    "model_name": "RandomForestRegressor",
    "target": "trip_duration_min",
    "task_type": "regression",
    "metrics": {
        "rmse": round(duration_rmse, 4),
        "r2": round(duration_r2, 4),
        "mae": round(duration_mae, 4),
        "mse": round(duration_mse, 4)
    },
    "execution_time_seconds": round(duration_execution_time, 2)
})

print("\nTrip duration prediction completed!")



Trip duration prediction completed!


### **5.3. Metrics**

#### View Data

In [19]:
print("\n==============================")
print(" MODEL RESULTS ")
print("==============================")

for result in model_results:

    print(f"\nTarget: {result['target']}")
    print(f"Model: {result['model_name']}")
    print(f"Task: {result['task_type']}")

    for metric_name, metric_value in result["metrics"].items():
        print(f"{metric_name}: {metric_value}")

    print(
        f"Execution Time: "
        f"{result['execution_time_seconds']} seconds"
    )


 MODEL RESULTS 

Target: total_amount
Model: RandomForestRegressor
Task: regression
rmse: 2.4122
r2: 0.9624
mae: 1.3509
mse: 5.8189
Execution Time: 814.22 seconds

Target: trip_duration_min
Model: RandomForestRegressor
Task: regression
rmse: 4.528
r2: 0.7967
mae: 2.6526
mse: 20.5025
Execution Time: 683.99 seconds


#### Save queries data

In [20]:
import json
import os
import subprocess
from datetime import datetime

new_run = {
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

    "spark_mode": MODE,

    "spark_config": scenarios[MODE],

    "model_results": model_results
}

json_filename = f"models_results_{USER}.json"

if os.path.exists(json_filename):

    with open(json_filename, "r") as f:

        try:
            history_data = json.load(f)

        except json.JSONDecodeError:
            history_data = {"runs": []}

else:
    history_data = {"runs": []}

if "runs" not in history_data:
    history_data["runs"] = []

history_data["runs"].append(new_run)

with open(json_filename, "w") as f:
    json.dump(history_data, f, indent=4)

print(f"\n[+] Results saved locally in '{json_filename}'!")
print(f"[i] Total executions stored: {len(history_data['runs'])}")

print("\n==============================")
print(" SAVED MODEL RESULTS ")
print("==============================")

for result in model_results:

    print(f"\nTarget: {result['target']}")
    print(f"Model: {result['model_name']}")
    print(f"Task: {result['task_type']}")

    for metric_name, metric_value in result["metrics"].items():
        print(f"{metric_name}: {metric_value}")

    print(
        f"Execution Time: "
        f"{result['execution_time_seconds']} seconds"
    )

print("\n==============================")

try:

    print(f"\n[~] Uploading results to Cloud Storage ({DATA_PATH})...")

    command = ["gsutil", "cp", json_filename, DATA_PATH]

    subprocess.run(
        command,
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    print("[+] Upload completed successfully!")

except subprocess.CalledProcessError as e:

    print("[-] Error uploading to Cloud Storage.")
    print(f"    Details: {e.stderr.decode('utf-8')}")

except NameError:

    print("[-] DATA_PATH variable not defined. Upload skipped.")


[+] Results saved locally in 'models_results_luana.json'!
[i] Total executions stored: 12

 SAVED MODEL RESULTS 

Target: total_amount
Model: RandomForestRegressor
Task: regression
rmse: 2.4122
r2: 0.9624
mae: 1.3509
mse: 5.8189
Execution Time: 814.22 seconds

Target: trip_duration_min
Model: RandomForestRegressor
Task: regression
rmse: 4.528
r2: 0.7967
mae: 2.6526
mse: 20.5025
Execution Time: 683.99 seconds


[~] Uploading results to Cloud Storage (gs://egd_feup/)...
[+] Upload completed successfully!


---

In [21]:
import json
import os

filename = f"model_results_{USER}.json"

if os.path.exists(filename):
    with open(filename, "r") as f:
        try:
            old_data = json.load(f)
        except json.JSONDecodeError:
            old_data = []
else:
    old_data = []

# garantir formato consistente
if not isinstance(old_data, list):
    old_data = []

# adicionar novos resultados
old_data.extend(model_results)

# guardar ficheiro atualizado
with open(filename, "w") as f:
    json.dump(old_data, f, indent=4)

print(f"[+] Model results saved locally in '{filename}'")

[+] Model results saved locally in 'model_results_luana.json'
